In [ ]:
import numpy as np
import pandas as pd

VERSION = "v6"
metadata_file1 = f"/biodata/franco/datasets/answerALS/metadata_{VERSION}/aals_dataportal_datatable_12062023.csv"
df = pd.read_csv(metadata_file1)


### Steps:
# get which samples have genomics and transcriptomics
# get for both the genotype sample id and the rnaseq sample id

In [ ]:
df

In [ ]:
df_wgs_rna = df[(df.HAS_GENOMICS == "Yes") & (df.HAS_TRANSCRIPTOMICS == "Yes")]

In [ ]:
import collections
metadata_files = f"/biodata/franco/datasets/answerALS/metadata_{VERSION}/aals_released_files.csv"
counter = 0
samples_dict = dict()
dup_dict = collections.defaultdict(int)
with open(metadata_files) as instream:
    next(instream)
    for line in instream:
        arr = line.split(",")
        donorid = arr[0]
        sampletype = arr[1]
        samplefile = arr[8]
        stagenum = arr[6]
        if stagenum == "2":
            sampleid = samplefile.split("/")[3]       
            counter += 1
            #print(donorid, sampletype, sampleid)
            if donorid not in samples_dict:
                samples_dict[donorid] = dict()
                samples_dict[donorid][sampletype] = sampleid
            elif sampletype not in samples_dict[donorid]:
                samples_dict[donorid][sampletype] = sampleid
            elif samples_dict[donorid][sampletype] != sampleid:
                print(donorid, sampletype, sampleid, samples_dict[donorid])
                dup_dict[f"{donorid}_{sampletype}"] += 1

In [ ]:
dup_dict

In [ ]:
# apply some fix to that repeated sample, like choose latest version?
## DECISION: DROP THIS SAMPLE

target = "CTRL-NEUEU392AE8"
# sample_dict = collections.defaultdict(list)
# with open(metadata_files) as instream:
#     next(instream)
#     for line in instream:
#         arr = line.split(",")
#         donorid = arr[0]
#         if donorid.startswith(target):
#             sampletype = arr[1]
#             if sampletype == "genomics":
#                 continue
#             samplefile = arr[8]
#             stagenum = arr[6]
#             if stagenum == "3":
#                 sampleid = samplefile.split("/")[3]
#                 filename = samplefile.split("/")[-1]
#                 print(donorid, sampletype, sampleid, filename)
#                 sample_dict[donorid].append(filename)

# for k in samples_dict:
#     if k != target:
#         print(samples_dict[k])

In [ ]:
df_samples = pd.DataFrame.from_dict(samples_dict)
print(df_samples.shape)
df_samples = df_samples.drop(columns=target)

df_samples = df_samples.T

In [ ]:
df_samples.index.name = "donor_id"

In [ ]:
df_samples.to_csv(f"/biodata/franco/datasets/answerALS/metadata_{VERSION}/all_donor_sample_ids.txt", sep="\t")